In [21]:
# importing libraries

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import Sequential
from tensorflow.keras.losses import BinaryCrossentropy

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [22]:
train_df = pd.read_pickle('data/intermediate_files/ranker_train_df.pkl')
val_df = pd.read_pickle('data/intermediate_files/ranker_validation_df.pkl')
test_df = pd.read_pickle('data/intermediate_files/ranker_test_df.pkl')

### Preprocessing

##### 1. Deterministic features

In [23]:
def preprocess_pre_sample(df):
    tp_df = df.copy()

    # Time features
    tp_df["query_time"] = pd.to_datetime(tp_df["query_time"])
    tp_df["day"] = tp_df["query_time"].dt.day_name().astype(str)
    tp_df["hour_of_day"] = tp_df["query_time"].dt.hour.astype(str)

    # Log features for skewed numeric columns
    for col in ["past_view_count", "past_atc_count", "past_order_count", "cg_score_raw"]:
        tp_df[col] = tp_df[col].fillna(0)
        tp_df[f"log_{col}"] = np.log1p(tp_df[col].astype(float))

    # Basic null handling
    tp_df["cg_score_norm"] = tp_df["cg_score_norm"].fillna(0)
    tp_df["user_history_bucket"] = tp_df["user_history_bucket"].fillna("unknown").astype(str)

    # Binary columns
    binary_base_cols = [
        "candidate_previously_purchased", "candidate_in_last_k_items",
        "from_user_co_interaction", "from_user_category", "from_user_generality"
    ]
    for col in binary_base_cols:
        if col in tp_df.columns:
            tp_df[col] = tp_df[col].fillna(0).astype("int8")

    # Rank bucket flags; NaN rank becomes 0 for every bucket
    rank_cols = ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    bucket_defs = {
        "1_10": (1, 10),
        "11_50": (11, 50),
        "51_100": (51, 100),
        "101_250": (101, 250),
        "251_500": (251, 500),
    }

    for col in rank_cols:
        if col not in tp_df.columns:
            continue
        for bucket_name, (lo, hi) in bucket_defs.items():
            tp_df[f"{col}_bucket_{bucket_name}"] = tp_df[col].between(lo, hi).fillna(False).astype("int8")

    return tp_df

##### 2. Negative Sampling

In [24]:

def sample_negatives_by_query(df, label_col="label_engagement", query_col="global_query_id", neg_per_pos=10, random_state=42):
    sampled_groups = []
    rng = np.random.default_rng(random_state)

    for _, group in df.groupby(query_col):
        pos = group[group[label_col] == 1]
        neg = group[group[label_col] == 0]

        # Skip all-negative queries for first ranker training
        if len(pos) == 0:
            continue

        n_neg = min(len(neg), len(pos) * neg_per_pos)

        if n_neg > 0:
            neg_sample = neg.sample(n=n_neg, random_state=int(rng.integers(0, 1_000_000)))
            sampled_groups.append(pd.concat([pos, neg_sample], axis=0))
        else:
            sampled_groups.append(pos)

    sampled_df = pd.concat(sampled_groups, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled_df

##### 3. Sanity Cleaning

In [25]:
def clean_numeric_features(df, feature_cols):
    df = df.copy()

    for col in feature_cols:
        df[col] = df[col].fillna(0).astype("float32")

    return df

In [26]:
# sampling validation/test to make it smaller, might comment this later

def sample_query_groups(df, n_queries, query_col='global_query_id', random_state=42):

    query_ids = df[query_col].drop_duplicates()
    sample_ids = query_ids.sample(n=min(n_queries, len(query_ids)), random_state=random_state)

    return df[df[query_col].isin(sample_ids)].copy()

In [27]:
## creating column name lists
label_col = 'label_engagement'

continuous_cols = ["log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_cg_score_raw", "cg_score_norm",]

# categorical_cols = ["user_history_bucket", "day", "hour_of_day"]

binary_cols = ["candidate_previously_purchased", "candidate_in_last_k_items", "from_user_co_interaction", "from_user_category",
    "from_user_generality"]

rank_bucket_cols = [
    f"{rank_col}_bucket_{bucket}"
    for rank_col in ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    for bucket in ["1_10", "11_50", "51_100", "101_250", "251_500"]
]

deep_feature_cols = continuous_cols+binary_cols+rank_bucket_cols

### Tensor Flow Data Set Creation

In [28]:
def make_tf_dataset(df, feature_cols, label_col, batch_size=512, shuffle=False, random_state=42):

    X = df[feature_cols].astype('float32').values
    y = df[label_col].astype('float32').values

    # creates tensorflow training input
    ds = tf.data.Dataset.from_tensor_slices((X,y))

    # shuffling the dataset using a rolling window of 1L rows at a time 
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 1_00_000), seed=random_state)

    # prepares data in form of batches, and also prefetch(tf.data.AUTOTUNE) improves 
    # performance by preparing the next training batch while the current training is happening
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

### Model Build

In [29]:
### Deep Ranker v0

def build_deep_ranker_v0(input_dim):

    model = Sequential([
        #prepare inputs
        layers.Input(shape=(input_dim,), name='feature_input'),
        layers.BatchNormalization(name='input_norm'),

        # hidden layers - num of hidden layers to be found out using val df
        layers.Dense(units=128, activation='relu', name='layer1'),

        # layers.Dense(units=128, use_bias=False, name='layer1_lin'),
        # layers.BatchNormalization(name='batch_norm_1'),
        # layers.Activation('relu', name='layer1_relu'),
        # layers.Dropout(0.2, name='dropout1'),

        layers.Dense(units=64, activation='relu', name='layer2'),
        # layers.Dense(units=64, use_bias=False, name='layer2_lin'),
        # layers.BatchNormalization(name='batch_norm_2'),
        # layers.Activation('relu', name='layer2_relu'),
        # layers.Dropout(0.2, name='dropout2'),

        layers.Dense(units=32, activation='relu', name='layer3'),
        # layers.Dense(units=32, use_bias=False, name='layer3_lin'),
        # layers.BatchNormalization(name='batch_norm_3'),
        # layers.Activation('relu', name='layer3_relu'),

        layers.Dense(units=1, activation='linear', name="engagement_score")
    ])

    return model


### Train Data & Validation data Creation

In [30]:
# train DS

neg_per_pos = 10
random_state = 42

# Deterministic features first
train_pre = preprocess_pre_sample(train_df)

# Negative sampling only on train
train_sampled = sample_negatives_by_query(
    train_pre,
    label_col=label_col,
    query_col="global_query_id",
    neg_per_pos=neg_per_pos,
    random_state=random_state
)

# getting all the relevant columns
deep_feature_cols = [c for c in deep_feature_cols if c in train_sampled.columns]

# final cleaning
train_model_df = clean_numeric_features(train_sampled, deep_feature_cols)

# creating tenssorflow datasets
train_ds = make_tf_dataset(train_model_df, deep_feature_cols, label_col, shuffle=True)

In [31]:
# Val DS

# sample queries from validation df
val_small = sample_query_groups(val_df, 10000, query_col='global_query_id', random_state=42)

# Deterministic features first
val_pre = preprocess_pre_sample(val_small)

# final cleaning
val_model_df = clean_numeric_features(val_pre, deep_feature_cols)

# creating tenssorflow datasets
val_ds = make_tf_dataset(val_model_df, deep_feature_cols, label_col, shuffle=False)


### Model initalisation

In [32]:
dnn_model_v0 = build_deep_ranker_v0(input_dim=len(deep_feature_cols))

dnn_model_v0.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(from_logits=True), 
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc', from_logits=True),
        tf.keras.metrics.AUC(curve='PR', name='pr_auc', from_logits=True)
    ]
    )

dnn_model_v0.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_norm (BatchNormalization) │ (None, 30)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer1 (Dense)                  │ (None, 128)            │         3,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Dense)                  │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ engagement_score (Dense)        │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,457 (56.47 KB)

 Trainable params: 14,397 (56.24 KB)

 Non-trainable params: 60 (240.00 B)

##### Fitting the model

In [33]:
trial_model_history = dnn_model_v0.fit(train_ds, epochs=20, validation_data=val_ds)

Epoch 1/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 18s 126ms/step - loss: 0.1858 - pr_auc: 0.6640 - roc_auc: 0.8926 - val_loss: 0.1266 - val_pr_auc: 0.0124 - val_roc_auc: 0.9098
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 14s 120ms/step - loss: 0.1417 - pr_auc: 0.7726 - roc_auc: 0.9295 - val_loss: 0.0615 - val_pr_auc: 0.0115 - val_roc_auc: 0.9352
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 14s 120ms/step - loss: 0.1403 - pr_auc: 0.7757 - roc_auc: 0.9312 - val_loss: 0.0518 - val_pr_auc: 0.0125 - val_roc_auc: 0.9371
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 14s 115ms/step - loss: 0.1392 - pr_auc: 0.7788 - roc_auc: 0.9327 - val_loss: 0.0520 - val_pr_auc: 0.0126 - val_roc_auc: 0.9318
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 15s 123ms/step - loss: 0.1393 - pr_auc: 0.7800 - roc_auc: 0.9321 - val_loss: 0.0503 - val_pr_auc: 0.0118 - val_roc_auc: 0.9353
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 13s 113ms/step - loss: 0.1391 - pr_auc: 0.7809 - roc_auc: 0.9324 - val_loss: 0.0429 - val_pr_auc: 0.0119 - val_roc_auc: 0.936

### Validation

In [34]:
def predict_scores_in_chunks(df, model, feature_cols, chunk_size=300_000, batch_size=512):
    scores = np.empty(len(df), dtype=np.float32)

    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))

        chunk = df.iloc[start:end].copy()
        chunk = preprocess_pre_sample(chunk)
        chunk = clean_numeric_features(chunk, feature_cols)

        X_chunk = chunk[feature_cols].astype('float32').values

        raw_preds = model.predict(X_chunk, batch_size=batch_size, verbose=0)

        preds = tf.nn.sigmoid(raw_preds).numpy().reshape(-1)

        scores[start:end] = preds.astype(np.float32)
        print(f"Scored rows {start:,} to {end:,}")

    return scores

### Evaluation Functions - Ranker

In [35]:
def evaluate_ranking(df, score_col, label_col="label_engagement", query_col="global_query_id",
                     ks=[10, 20, 50], mode="reranker_only"):
    eval_df = df.copy()

    if mode == "reranker_only":
        
        q_pos = eval_df.groupby(query_col)[label_col].sum()
        keep_qids = q_pos[q_pos > 0].index
        eval_df = eval_df[eval_df[query_col].isin(keep_qids)].copy()

    eval_df = eval_df.sort_values([query_col, score_col, "cg_rank"], ascending=[True, False, True])

    rows = []
    n_queries = eval_df[query_col].nunique()

    for k in ks:
        hit_list, recall_list, ndcg_list, mrr_list = [], [], [], []

        for _, g in eval_df.groupby(query_col, sort=False):
            labels = g[label_col].to_numpy()
            total_pos = labels.sum()

            if total_pos == 0:
                hit_list.append(0.0)
                recall_list.append(0.0)
                ndcg_list.append(0.0)
                mrr_list.append(0.0)
                continue

            topk = labels[:k]
            hits = topk.sum()

            hit = 1.0 if hits > 0 else 0.0
            recall = hits / total_pos

            discounts = 1.0 / np.log2(np.arange(2, len(topk) + 2))
            dcg = np.sum(topk * discounts)

            ideal_hits = int(min(total_pos, k))
            ideal_discounts = 1.0 / np.log2(np.arange(2, ideal_hits + 2))
            idcg = np.sum(ideal_discounts)
            ndcg = dcg / idcg if idcg > 0 else 0.0

            pos_idx = np.where(topk == 1)[0]
            mrr = 1.0 / (pos_idx[0] + 1) if len(pos_idx) > 0 else 0.0

            hit_list.append(hit)
            recall_list.append(recall)
            ndcg_list.append(ndcg)
            mrr_list.append(mrr)

        rows.append({
            "score_col": score_col,
            "mode": mode,
            "K": k,
            "n_queries": n_queries,
            "HitRate": np.mean(hit_list),
            "Recall": np.mean(recall_list),
            "NDCG": np.mean(ndcg_list),
            "MRR": np.mean(mrr_list),
        })

    return pd.DataFrame(rows)

In [36]:
def add_lift(compare_df, baseline_score_col="cg_baseline_score", model_score_col="lr_score"):
    base = compare_df[compare_df["score_col"] == baseline_score_col].copy()
    model = compare_df[compare_df["score_col"] == model_score_col].copy()

    merged = model.merge(
        base,
        on=["mode", "K"],
        suffixes=("_model", "_baseline")
    )

    for metric in ["HitRate", "Recall", "NDCG", "MRR"]:
        merged[f"{metric}_lift_abs"] = merged[f"{metric}_model"] - merged[f"{metric}_baseline"]
        merged[f"{metric}_lift_pct"] = np.where(
            merged[f"{metric}_baseline"] > 0,
            100 * merged[f"{metric}_lift_abs"] / merged[f"{metric}_baseline"],
            np.nan
        )

    return merged

#### Evaluation Queries

In [37]:
# validation data

val_scored = val_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
val_scored['dnn_v0_score'] = predict_scores_in_chunks(val_small, dnn_model_v0, deep_feature_cols)

val_scored["cg_baseline_score"] = -val_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
val_cg_rerank = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

val_dnn_rerank = evaluate_ranking(
    val_scored,
    score_col="dnn_v0_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
val_cg_all = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

val_dnn_all = evaluate_ranking(
    val_scored,
    score_col="dnn_v0_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# val dfs

val_compare_rerank = pd.concat([val_cg_rerank, val_dnn_rerank], ignore_index=True)
val_compare_all = pd.concat([val_cg_all, val_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,010,291


In [38]:
# test data
test_small = sample_query_groups(test_df, 10000, query_col='global_query_id', random_state=42)

test_scored = test_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
test_scored['dnn_v0_score'] = predict_scores_in_chunks(test_small, dnn_model_v0, deep_feature_cols)

test_scored["cg_baseline_score"] = -test_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
test_cg_rerank = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

test_dnn_rerank = evaluate_ranking(
    test_scored,
    score_col="dnn_v0_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
test_cg_all = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

test_dnn_all = evaluate_ranking(
    test_scored,
    score_col="dnn_v0_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# test dfs

test_compare_rerank = pd.concat([test_cg_rerank, test_dnn_rerank], ignore_index=True)
test_compare_all = pd.concat([test_cg_all, test_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,065,555


In [39]:
val_lift_rerank = add_lift(val_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="dnn_v0_score")
val_lift_all = add_lift(val_compare_all, baseline_score_col="cg_baseline_score", model_score_col="dnn_v0_score")

val_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.163966,0.563625,0.399659,243.744389,0.314721,0.715736,0.401015,127.419355
1,reranker_only,20,0.194895,0.575326,0.380432,195.198824,0.436548,0.756345,0.319797,73.255814
2,reranker_only,50,0.241261,0.590721,0.349460,144.847317,0.659898,0.827411,0.167513,25.384615
3,reranker_only,100,0.275667,0.600182,0.324514,117.719494,0.847716,0.873096,0.025381,2.994012


In [40]:
test_lift_rerank = add_lift(test_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="dnn_v0_score")
test_lift_all = add_lift(test_compare_all, baseline_score_col="cg_baseline_score", model_score_col="dnn_v0_score")

test_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.156688,0.530644,0.373956,238.663795,0.276836,0.723164,0.446328,161.224490
1,reranker_only,20,0.184515,0.550647,0.366132,198.428942,0.389831,0.790960,0.401130,102.898551
2,reranker_only,50,0.226430,0.563924,0.337494,149.050048,0.593220,0.847458,0.254237,42.857143
3,reranker_only,100,0.268702,0.574945,0.306243,113.971345,0.836158,0.903955,0.067797,8.108108
